# TrendPulse Part 2 — NumPy & Pandas
**Project:** TrendPulse — YouTube Trending Video Statistics  
**Purpose:** Build a transparent engagement score, clean and combine country datasets, and analyze trending patterns.

## How to run this notebook
1. Keep this notebook in `part2_numpy_pandas/`.
2. Ensure the committed country CSV files and `categories.csv` are in `part2_numpy_pandas/data/`.
3. In Google Colab, open the repository folder (or upload the notebook and its `data` folder). Set `DATA_DIR` in the setup cell if your folder is in a different location.
4. Use **Runtime → Run all**. This notebook does not call an external API or download data.
5. Read the Markdown interpretation notes and update them if your actual results suggest a more precise business conclusion.

**Important:** The dataset's actual data-quality findings and numerical outputs are generated by the notebook. Do not replace them with invented values.

In [1]:

from google.colab import files
from pathlib import Path

# Create the local Part 2 data folder
DATA_DIR = Path("/content/part2_numpy_pandas/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Select the CSV files from your Windows computer
uploaded = files.upload()

# Save the selected files into the data folder
for filename, content in uploaded.items():
    (DATA_DIR / filename).write_bytes(content)

print("Files saved successfully:")
for path in sorted(DATA_DIR.glob("*.csv")):
    print(path.name)


Saving categories.csv to categories (1).csv
Saving GBvideos.csv to GBvideos (1).csv
Saving INvideos.csv to INvideos (1).csv
Saving USvideos.csv to USvideos (1).csv
Files saved successfully:
GBvideos (1).csv
GBvideos.csv
INvideos (1).csv
INvideos.csv
USvideos (1).csv
USvideos.csv
categories (1).csv
categories.csv


## 0. Setup and configuration
We use NumPy and Pandas for analysis. The setup searches a few likely local folder paths so the notebook can run from the repository root, from inside `part2_numpy_pandas`, or in a Colab working directory where the data folder was uploaded.

In [22]:
from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)
warnings.filterwarnings("ignore", category=FutureWarning)

POSSIBLE_DATA_DIRS = [
    Path("data"),
    Path("part2_numpy_pandas") / "data",
    Path("part2_numpy_pandas") / "part2_numpy_pandas" / "data",
    Path("/content/part2_numpy_pandas/data"),
    Path("/content/part2_numpy_pandas/part2_numpy_pandas/data"),
    Path("/content/part2_numpy_pandas/data"),

]

DATA_DIR = next((p for p in POSSIBLE_DATA_DIRS if p.exists() and list(p.glob("*.csv"))), None)

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find CSV files. Put the notebook beside a 'data' folder, "
        "or set DATA_DIR manually to the folder containing the country CSV files and categories.csv."
    )

print("Using data folder:", DATA_DIR.resolve())
print("CSV files found:")
for file_path in sorted(DATA_DIR.glob("*.csv")):
    print(" -", file_path.name)

Using data folder: /content/part2_numpy_pandas/data
CSV files found:
 - GBvideos (1).csv
 - GBvideos.csv
 - INvideos (1).csv
 - INvideos.csv
 - USvideos (1).csv
 - USvideos.csv
 - categories (1).csv
 - categories.csv


## 1. Load the country data
Each country file is loaded from the local `data/` folder. We add a `country` column using the filename (for example, `GBvideos.csv` → `GB`). This section uses `pd.read_csv()` for loading.

In [23]:
csv_files = sorted(
    p for p in DATA_DIR.glob("*videos.csv")
    if p.name.lower() != "categories.csv"
)

if not csv_files:
    # More permissive fallback for country CSV filenames
    csv_files = sorted(
        p for p in DATA_DIR.glob("*.csv")
        if p.name.lower() != "categories.csv"
    )

if not csv_files:
    raise FileNotFoundError(f"No country video CSV files found in {DATA_DIR}")

country_dfs = {}
load_errors = {}

for file_path in csv_files:
    country_code = file_path.stem.replace("videos", "").strip("_- ").upper()
    try:
        country_df = pd.read_csv(file_path, low_memory=False)
        country_df["country"] = country_code
        country_dfs[country_code] = country_df
        print(f"{file_path.name}: loaded {country_df.shape[0]:,} rows × {country_df.shape[1]:,} columns")
    except Exception as exc:
        load_errors[file_path.name] = str(exc)
        print(f"Could not load {file_path.name}: {exc}")

if not country_dfs:
    raise ValueError("No country CSV was loaded successfully.")

print("\nCountry codes loaded:", sorted(country_dfs))
if load_errors:
    print("Files with loading errors:", load_errors)

GBvideos.csv: loaded 15,075 rows × 17 columns
INvideos.csv: loaded 15,075 rows × 17 columns
USvideos.csv: loaded 16,080 rows × 17 columns

Country codes loaded: ['GB', 'IN', 'US']


## 2. Inspect each country DataFrame
The next cell displays `head()`, `info()`, `describe()` and `shape` for every country DataFrame. `info()` prints its summary and returns `None`.

In [4]:
for country_code, frame in country_dfs.items():
    print("\n" + "=" * 90)
    print(f"COUNTRY: {country_code}")
    print("SHAPE:", frame.shape)
    display(frame.head())
    print("\nINFO:")
    frame.info()
    print("\nDESCRIBE (numeric columns):")
    display(frame.describe(include="all").transpose())


COUNTRY: GB
SHAPE: (15075, 17)


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
0,u_nKn025TZS,17.14.11,Wild Winter Prank,OfficialRecipe78,28,2017-10-18T07:42:02.000Z,[none],84344,2155.0,163,475.0,https://i.ytimg.com/vi/u_nKn025TZS/default.jpg,False,False,False,Watch this video. Subscribe for more content about budget. Business enquiries in the channel abo...,GB
1,FoQAEGxMMQD,17.14.11,Wild Cricket Tutorial FINALE,FinalTrailer97,26,2017-11-14T12:56:18.000Z,Minecraft|Paris,230434,5412.0,330,1704.0,https://i.ytimg.com/vi/FoQAEGxMMQD/default.jpg,False,False,False,Watch this video. Subscribe for more content about monsoon. Business enquiries in the channel ab...,GB
2,wHqDBuStXq3,17.14.11,Epic Space Challenge,UltimateReview54,23,2017-11-13T06:19:10.000Z,Paris,338852,16813.0,1333,1385.0,https://i.ytimg.com/vi/wHqDBuStXq3/default.jpg,False,False,False,Watch this video. Subscribe for more content about minecraft. Business enquiries in the channel ...,GB
3,ZId0IGzSaaZ,17.14.11,Quiet Paris Prank LIVE?!,FinalUpdate79,24,2017-11-14T11:37:33.000Z,Phones,160896,4221.0,483,1342.0,https://i.ytimg.com/vi/ZId0IGzSaaZ/default.jpg,False,False,False,Watch this video. Subscribe for more content about festival. Business enquiries in the channel a...,GB
4,1-xA4eRrYSH,17.14.11,Nova Kane - Chess (Official Video)!,QuietInterview9,10,2017-11-14T03:13:26.000Z,Chess|Minecraft|Festival|Guitar,59441,1475.0,77,60.0,https://i.ytimg.com/vi/1-xA4eRrYSH/default.jpg,False,False,False,Watch this video. Subscribe for more content about cats. Business enquiries in the channel about...,GB



INFO:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15075 entries, 0 to 15074
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   video_id                15075 non-null  object 
 1   trending_date           15075 non-null  object 
 2   title                   15075 non-null  object 
 3   channel_title           15075 non-null  object 
 4   category_id             15075 non-null  int64  
 5   publish_time            15075 non-null  object 
 6   tags                    15075 non-null  object 
 7   views                   15075 non-null  int64  
 8   likes                   15071 non-null  float64
 9   dislikes                15075 non-null  int64  
 10  comment_count           15063 non-null  float64
 11  thumbnail_link          15075 non-null  object 
 12  comments_disabled       15075 non-null  bool   
 13  ratings_disabled        15075 non-null  bool   
 14  video_error_or_removed  15075 n

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
video_id,15075,4113,O6zdCRwWdf1,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trending_date,15075,75,17.22.11,203,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,15075,3629,Kairo - Chess (Official Video),44,NaN,NaN,NaN,NaN,NaN,NaN,NaN
channel_title,15075,387,QuietChallenge75,1224,NaN,NaN,NaN,NaN,NaN,NaN,NaN
category_id,15075.0,NaN,NaN,NaN,17.102023,7.749379,1.0,10.0,20.0,24.0,28.0
publish_time,15075,4113,2017-12-11T04:42:12.000Z,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tags,15075,2147,[none],1412,NaN,NaN,NaN,NaN,NaN,NaN,NaN
views,15075.0,NaN,NaN,NaN,1580015.492537,9105598.442089,1066.0,121176.5,316900.0,867429.5,360760164.0
likes,15071.0,NaN,NaN,NaN,65096.999602,476940.047152,0.0,3658.0,10537.0,31800.0,27537822.0
dislikes,15075.0,NaN,NaN,NaN,2938.435091,15407.806631,0.0,194.0,569.0,1688.5,618175.0



COUNTRY: IN
SHAPE: (15075, 17)


...

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
video_id,15075,5878,aGeu6FBmeRM,23,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trending_date,15075,75,17.18.11,203,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,15075,4990,Rook & Vale - Festival (Live),31,NaN,NaN,NaN,NaN,NaN,NaN,NaN
channel_title,15075,408,OfficialReview41,659,NaN,NaN,NaN,NaN,NaN,NaN,NaN
category_id,15075.0,NaN,NaN,NaN,19.836086,7.341271,1.0,17.0,24.0,24.0,28.0
publish_time,15075,5874,2017-11-16T09:34:58.000Z,23,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tags,15075,2833,[none],1209,NaN,NaN,NaN,NaN,NaN,NaN,NaN
views,15075.0,NaN,NaN,NaN,954592.446567,3035646.780561,3542.0,152165.5,355802.0,863998.0,123208090.0
likes,15071.0,NaN,NaN,NaN,33218.333289,104210.317248,0.0,4107.0,10915.0,29262.0,3634665.0
dislikes,15075.0,NaN,NaN,NaN,1937.757877,5791.482855,0.0,257.0,681.0,1815.5,287063.0



COUNTRY: US
SHAPE: (16080, 17)


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
0,g_N-w8LeAi8,17.14.11,Honest Football Prank LEAKED,QuietSpeech10,22,2017-11-14T07:04:28.000Z,Monsoon|Robots,48555,2189.0,180,136.0,https://i.ytimg.com/vi/g_N-w8LeAi8/default.jpg,False,False,False,Watch this video. Subscribe for more content about chess. Business enquiries in the channel abou...,US
1,Ugyql2NlPAd,17.14.11,Surprising Cricket Review,IncredibleChallenge41,25,2017-11-12T22:35:48.000Z,Winter,198385,1159.0,492,1446.0,https://i.ytimg.com/vi/Ugyql2NlPAd/default.jpg,False,False,False,Watch this video. Subscribe for more content about bangalore. Business enquiries in the channel ...,US
2,8fE19jkxJ9n,17.14.11,Mira Solis - Football (Live) OFFICIAL,WildTutorial42,10,2017-11-13T08:40:22.000Z,Cricket,231656,8742.0,896,432.0,https://i.ytimg.com/vi/8fE19jkxJ9n/default.jpg,False,False,False,Watch this video. Subscribe for more content about election. Business enquiries in the channel a...,US
3,WOY0nn7HCsO,17.14.11,Zed Rivera - Space (Lyric Video)?!,ShockingInterview30,10,2017-10-27T22:21:47.000Z,Festival|Guitar,8083,176.0,8,19.0,https://i.ytimg.com/vi/WOY0nn7HCsO/default.jpg,False,False,False,Watch this video. Subscribe for more content about phones. Business enquiries in the channel abo...,US
4,2Lc_Cs5VVr_,17.14.11,Surprising Election Recipe,UltimateVlog96,27,2017-11-09T13:20:20.000Z,Festival,373161,11305.0,167,891.0,https://i.ytimg.com/vi/2Lc_Cs5VVr_/default.jpg,False,False,False,Watch this video. Subscribe for more content about guitar. Business enquiries in the channel abo...,US



INFO:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16080 entries, 0 to 16079
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   video_id                16080 non-null  object 
 1   trending_date           16080 non-null  object 
 2   title                   16080 non-null  object 
 3   channel_title           16080 non-null  object 
 4   category_id             16080 non-null  int64  
 5   publish_time            16080 non-null  object 
 6   tags                    16080 non-null  object 
 7   views                   16080 non-null  int64  
 8   likes                   16076 non-null  float64
 9   dislikes                16080 non-null  int64  
 10  comment_count           16068 non-null  float64
 11  thumbnail_link          16080 non-null  object 
 12  comments_disabled       16080 non-null  bool   
 13  ratings_disabled        16080 non-null  bool   
 14  video_error_or_removed  16080 n

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
video_id,16080,5068,BsTMdCTYs7P,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trending_date,16080,80,17.30.12,205,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,16080,4403,Sunny Park - Bangalore (Live),33,NaN,NaN,NaN,NaN,NaN,NaN,NaN
channel_title,16080,408,QuietChallenge75,644,NaN,NaN,NaN,NaN,NaN,NaN,NaN
category_id,16080.0,NaN,NaN,NaN,18.975871,7.590729,1.0,10.0,23.0,24.0,28.0
publish_time,16080,5064,2018-01-01T10:45:28.000Z,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tags,16080,2501,[none],1164,NaN,NaN,NaN,NaN,NaN,NaN,NaN
views,16080.0,NaN,NaN,NaN,1012289.201741,3746760.856871,1621.0,117307.75,289072.5,750719.0,129229245.0
likes,16076.0,NaN,NaN,NaN,39504.637534,171941.394484,0.0,3353.0,9229.0,26267.0,6795138.0
dislikes,16080.0,NaN,NaN,NaN,2115.772015,7973.92775,0.0,194.0,540.0,1506.0,247546.0


## 3. Combine country records with `concat()`
This combines the country DataFrames into one DataFrame. The source country frames remain available in `country_dfs`.

In [24]:
combined_raw = pd.concat(country_dfs.values(), ignore_index=True, sort=False)
print("Combined raw shape:", combined_raw.shape)
display(combined_raw.head())
print("\nCombined info:")
combined_raw.info()
print("\nCombined describe:")
display(combined_raw.describe(include="all").transpose())

Combined raw shape: (46230, 17)


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
0,u_nKn025TZS,17.14.11,Wild Winter Prank,OfficialRecipe78,28,2017-10-18T07:42:02.000Z,[none],84344,2155.0,163,475.0,https://i.ytimg.com/vi/u_nKn025TZS/default.jpg,False,False,False,Watch this video. Subscribe for more content about budget. Business enquiries in the channel abo...,GB
1,FoQAEGxMMQD,17.14.11,Wild Cricket Tutorial FINALE,FinalTrailer97,26,2017-11-14T12:56:18.000Z,Minecraft|Paris,230434,5412.0,330,1704.0,https://i.ytimg.com/vi/FoQAEGxMMQD/default.jpg,False,False,False,Watch this video. Subscribe for more content about monsoon. Business enquiries in the channel ab...,GB
2,wHqDBuStXq3,17.14.11,Epic Space Challenge,UltimateReview54,23,2017-11-13T06:19:10.000Z,Paris,338852,16813.0,1333,1385.0,https://i.ytimg.com/vi/wHqDBuStXq3/default.jpg,False,False,False,Watch this video. Subscribe for more content about minecraft. Business enquiries in the channel ...,GB
3,ZId0IGzSaaZ,17.14.11,Quiet Paris Prank LIVE?!,FinalUpdate79,24,2017-11-14T11:37:33.000Z,Phones,160896,4221.0,483,1342.0,https://i.ytimg.com/vi/ZId0IGzSaaZ/default.jpg,False,False,False,Watch this video. Subscribe for more content about festival. Business enquiries in the channel a...,GB
4,1-xA4eRrYSH,17.14.11,Nova Kane - Chess (Official Video)!,QuietInterview9,10,2017-11-14T03:13:26.000Z,Chess|Minecraft|Festival|Guitar,59441,1475.0,77,60.0,https://i.ytimg.com/vi/1-xA4eRrYSH/default.jpg,False,False,False,Watch this video. Subscribe for more content about cats. Business enquiries in the channel about...,GB



Combined info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46230 entries, 0 to 46229
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   video_id                46230 non-null  object 
 1   trending_date           46230 non-null  object 
 2   title                   46230 non-null  object 
 3   channel_title           46230 non-null  object 
 4   category_id             46230 non-null  int64  
 5   publish_time            46230 non-null  object 
 6   tags                    46230 non-null  object 
 7   views                   46230 non-null  int64  
 8   likes                   46218 non-null  float64
 9   dislikes                46230 non-null  int64  
 10  comment_count           46194 non-null  float64
 11  thumbnail_link          46230 non-null  object 
 12  comments_disabled       46230 non-null  bool   
 13  ratings_disabled        46230 non-null  bool   
 14  video_error_or_removed

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
video_id,46230,14699,iAvRHGeUpBR,28,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trending_date,46230,80,17.30.12,608,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,46230,10404,Sunny Park - Bangalore (Live),66,NaN,NaN,NaN,NaN,NaN,NaN,NaN
channel_title,46230,438,QuietChallenge75,2464,NaN,NaN,NaN,NaN,NaN,NaN,NaN
category_id,46230.0,NaN,NaN,NaN,18.645339,7.646745,1.0,10.0,22.0,24.0,28.0
publish_time,46230,14678,2017-11-27T11:50:56.000Z,28,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tags,46230,6056,[none],3785,NaN,NaN,NaN,NaN,NaN,NaN,NaN
views,46230.0,NaN,NaN,NaN,1178603.180748,5916188.240095,1066.0,129463.25,318277.5,824050.5,360760164.0
likes,46218.0,NaN,NaN,NaN,45800.054849,296955.393069,0.0,3656.0,10201.0,28987.75,27537822.0
dislikes,46230.0,NaN,NaN,NaN,2325.983625,10518.962756,0.0,213.0,594.0,1666.0,618175.0


## 4. Inspect columns and prepare data types
The common YouTube trending dataset usually contains fields such as `video_id`, `title`, `channel_title`, `category_id`, `publish_time`, `trending_date`, `views`, `likes`, `dislikes`, and `comment_count`. This cell handles fields that are present and reports absent expected fields.

In [25]:
expected_columns = [
    "video_id", "title", "channel_title", "category_id", "publish_time",
    "trending_date", "views", "likes", "dislikes", "comment_count", "country"
]
missing_expected = [col for col in expected_columns if col not in combined_raw.columns]
print("Expected columns not present in this dataset:", missing_expected)

combined = combined_raw.copy()

numeric_columns = ["views", "likes", "dislikes", "comment_count", "category_id"]
for col in numeric_columns:
    if col in combined.columns:
        combined[col] = pd.to_numeric(combined[col], errors="coerce")

if "publish_time" in combined.columns:
    combined["publish_time"] = pd.to_datetime(combined["publish_time"], errors="coerce", utc=True)

def parse_trending_date(value):
    """Parse common YouTube trending-date formats, including YY.DD.MM."""
    if pd.isna(value):
        return pd.NaT
    text_value = str(value).strip()
    # Original Kaggle files commonly use YY.DD.MM, e.g. 17.14.11.
    if re.fullmatch(r"\d{2}\.\d{2}\.\d{2}", text_value):
        yy, dd, mm = text_value.split(".")
        year = 2000 + int(yy) if int(yy) < 70 else 1900 + int(yy)
        try:
            return pd.Timestamp(year=year, month=int(mm), day=int(dd))
        except ValueError:
            return pd.NaT
    return pd.to_datetime(text_value, errors="coerce", utc=True)

if "trending_date" in combined.columns:
    combined["trending_date"] = combined["trending_date"].apply(parse_trending_date)
    # Normalize to timezone-naive calendar dates for day-level analysis.
    combined["trending_date"] = pd.to_datetime(combined["trending_date"], errors="coerce").dt.tz_localize(None).dt.normalize()

for col in ["title", "channel_title", "video_id", "description", "tags"]:
    if col in combined.columns:
        combined[col] = combined[col].astype("string")

print("Prepared combined shape:", combined.shape)
print("Data types:")
display(combined.dtypes.rename("dtype").to_frame())

Expected columns not present in this dataset: []
Prepared combined shape: (46230, 17)
Data types:


,dtype
video_id,string[python]
trending_date,datetime64[ns]
title,string[python]
channel_title,string[python]
category_id,int64
publish_time,"datetime64[ns, UTC]"
tags,string[python]
views,int64
likes,float64
dislikes,int64


## 5. Data-quality audit: missing values, invalid dates and removed/errored videos
We report missing counts and percentages for every column. The common dataset may encode removed videos or unavailable titles/descriptions with text such as `[Deleted video]`, `[Private video]`, or `[Channel does not exist]`; we flag these records instead of silently assuming every row is a normal public video.

In [26]:
missing_count = combined.isna().sum()
missing_pct = (missing_count / len(combined) * 100).round(2) if len(combined) else missing_count.astype(float)
missing_report = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percent": missing_pct
}).query("missing_count > 0").sort_values("missing_count", ascending=False)

print("Missing-value report (only columns with missing values):")
display(missing_report if not missing_report.empty else pd.DataFrame(columns=["missing_count", "missing_percent"]))

removed_patterns = [
    r"\[deleted video\]",
    r"\[private video\]",
    r"\[channel does not exist\]",
    r"\[unavailable video\]",
]
text_cols = [c for c in ["title", "description", "channel_title"] if c in combined.columns]
if text_cols:
    removed_mask = pd.Series(False, index=combined.index)
    for col in text_cols:
        removed_mask |= combined[col].astype("string").str.contains("|".join(removed_patterns), case=False, na=False, regex=True)
else:
    removed_mask = pd.Series(False, index=combined.index)

print("Rows flagged as removed/private/unavailable by text patterns:", int(removed_mask.sum()))
display(combined.loc[removed_mask, [c for c in ["country", "video_id", "title", "channel_title", "trending_date"] if c in combined.columns]].head(20))

if "trending_date" in combined.columns:
    print("Rows with invalid/missing trending_date:", int(combined["trending_date"].isna().sum()))
if "publish_time" in combined.columns:
    print("Rows with invalid/missing publish_time:", int(combined["publish_time"].isna().sum()))

Missing-value report (only columns with missing values):


,missing_count,missing_percent
description,1624,3.51
comment_count,36,0.08
likes,12,0.03


Rows flagged as removed/private/unavailable by text patterns: 0


,country,video_id,title,channel_title,trending_date


Rows with invalid/missing trending_date: 0
Rows with invalid/missing publish_time: 0


### Cleaning decisions (transparent and reproducible)
- **Missing-value strategy:** numeric engagement fields with missing values are treated as unknown, not automatically as zero. Score calculations use safe numeric arrays and fill missing normalized features with zero only for scoring, which means “no usable measured contribution” rather than a claim that the true value was zero.
- **Text fields:** missing descriptions/titles are reported. We do not invent text to fill them.
- **Dates:** unparseable dates are reported and excluded only from analyses that require the corresponding date.
- **Removed/private videos:** flagged for review. We keep the original rows in the audit table so the number is visible; later analyses can exclude flagged rows when appropriate.
- **Duplicates:** the primary duplicate definition is the same `video_id` + `trending_date` + `country`, because a video can legitimately appear on multiple trending dates or in different countries.
- **Negative numeric counts:** checked below; any found values are reported and excluded from score features by treating them as missing.
Update this section if your dataset reveals additional issues.

In [27]:
negative_count_report = {}
for col in ["views", "likes", "dislikes", "comment_count"]:
    if col in combined.columns:
        negative_count_report[col] = int((combined[col] < 0).sum())
print("Negative counts by numeric field:", negative_count_report)

if all(c in combined.columns for c in ["video_id", "trending_date", "country"]):
    duplicate_key = ["video_id", "trending_date", "country"]
    duplicate_mask = combined.duplicated(subset=duplicate_key, keep="first")
else:
    duplicate_key = None
    duplicate_mask = combined.duplicated(keep="first")

duplicate_rows_removed = int(duplicate_mask.sum())
print("Duplicate definition:", duplicate_key if duplicate_key else "identical full rows")
print("Duplicate rows identified for removal:", duplicate_rows_removed)

clean = combined.loc[~duplicate_mask].copy()
print("Shape after duplicate removal:", clean.shape)

Negative counts by numeric field: {'views': 0, 'likes': 0, 'dislikes': 0, 'comment_count': 0}
Duplicate definition: ['video_id', 'trending_date', 'country']
Duplicate rows identified for removal: 230
Shape after duplicate removal: (46000, 17)


## 6. Handle missing values
This section demonstrates `isna()`, `fillna()`, `dropna()` and `drop_duplicates()`. We preserve `clean` as the main cleaned dataset. A separate `analysis` frame excludes records whose title indicates a removed/private/unavailable video. Missing descriptions do not by themselves remove a record, because descriptions are not needed for most numerical analyses.

In [9]:
# Demonstrate fillna(): use a separate scoring/analysis copy, preserving clean.
analysis = clean.copy()

for col in ["views", "likes", "dislikes", "comment_count"]:
    if col in analysis.columns:
        analysis.loc[analysis[col] < 0, col] = np.nan

# Demonstrate dropna() for date-dependent tasks in a separate view.
date_required = [c for c in ["publish_time", "trending_date"] if c in analysis.columns]
date_analysis = analysis.dropna(subset=date_required).copy() if date_required else analysis.copy()

# Exclude records clearly marked as removed/private/unavailable from main performance analysis.
if text_cols:
    analysis_removed_mask = pd.Series(False, index=analysis.index)
    for col in text_cols:
        analysis_removed_mask |= analysis[col].astype("string").str.contains("|".join(removed_patterns), case=False, na=False, regex=True)
    analysis = analysis.loc[~analysis_removed_mask].copy()

print("Rows after duplicate removal:", len(clean))
print("Rows in performance analysis after removed/private flags:", len(analysis))
print("Rows with valid dates for date-dependent analysis:", len(date_analysis))

# Demonstrate fillna() for text-only display convenience; do not overwrite original clean data.
if "description" in analysis.columns:
    analysis["description_for_display"] = analysis["description"].fillna("[Missing description]")
    print("Missing descriptions remain documented; a display-only column uses a placeholder.")

Rows after duplicate removal: 46000
Rows in performance analysis after removed/private flags: 46000
Rows with valid dates for date-dependent analysis: 46000
Missing descriptions remain documented; a display-only column uses a placeholder.


## 7. Task 1 — NumPy arrays: shape, dtype, indexing and slicing
The required four arrays are created from the cleaned performance-analysis data. `pd.to_numeric(..., errors='coerce')` converts invalid entries to missing values, represented as `NaN` in these arrays.

In [10]:
def numeric_array(frame, column):
    if column not in frame.columns:
        raise KeyError(f"Required column '{column}' is missing from the combined dataset.")
    return pd.to_numeric(frame[column], errors="coerce").to_numpy(dtype=float)

views = numeric_array(analysis, "views")
likes = numeric_array(analysis, "likes")
dislikes = numeric_array(analysis, "dislikes")
comment_count = numeric_array(analysis, "comment_count")

arrays = {
    "views": views,
    "likes": likes,
    "dislikes": dislikes,
    "comment_count": comment_count
}
for name, arr in arrays.items():
    print(f"{name}: shape={arr.shape}, dtype={arr.dtype}")

print("\nFirst 10 view counts:", views[:10])
print("Every 100th view count:", views[::100])
print("Chosen sub-array (indices 20–29):", views[20:30])

valid_views = views[np.isfinite(views)]
print("\n10 largest view counts:")
print(np.sort(valid_views)[-10:][::-1] if valid_views.size else "No valid view counts")

views: shape=(46000,), dtype=float64
likes: shape=(46000,), dtype=float64
dislikes: shape=(46000,), dtype=float64
comment_count: shape=(46000,), dtype=float64

First 10 view counts: [ 84344. 230434. 338852. 160896.  59441.  91008. 121219. 485512. 317538.
 315666.]
Every 100th view count: [8.43440000e+04 6.56734000e+05 1.09438000e+05 2.89683000e+05
 1.24665000e+05 6.76490000e+04 1.56685000e+05 7.99240000e+04
 1.95161000e+05 9.08364200e+06 2.28949000e+05 1.39395000e+05
 2.85171000e+05 4.80365000e+05 3.46544000e+05 3.83060000e+04
 1.88850200e+06 1.51220000e+04 2.05951600e+06 3.50490000e+04
 2.74967500e+06 3.65810000e+04 2.87239200e+06 2.62046000e+05
 3.30825200e+06 2.08771000e+05 4.20106600e+06 3.67155000e+05
 4.84398400e+06 4.19930000e+04 6.10680500e+06 1.15155700e+06
 7.22438300e+06 3.38165000e+05 8.98173000e+06 5.50084100e+06
 1.07819710e+07 1.25886000e+05 1.29070610e+07 1.74729000e+05
 1.65517480e+07 1.54008000e+05 2.02488650e+07 8.47543000e+05
 2.38089240e+07 1.50376500e+06 2.9473048

## 8. Task 2 — Vectorized engagement features (no explicit loops)
### Engagement-score weights
We use a transparent weighted score on a 0–1 scale:
- **35% normalized views:** reach is the primary signal of audience size.
- **25% normalized likes:** likes indicate positive audience response.
- **20% normalized comments:** comments capture active discussion.
- **10% like ratio:** likes divided by likes + dislikes, when the denominator is positive.
- **10% normalized comment rate:** comments divided by views, which adjusts discussion for audience size.

These are practical starting weights, not universal truths. Views are weighted most because this is a trending-video dataset, while response and interaction rates add quality/context. Min-max scaling makes the three count-based features comparable, but is sensitive to extreme outliers; log-scaled views are calculated separately for visualization.

In [11]:
def safe_numeric_series(frame, column):
    return pd.to_numeric(frame[column], errors="coerce").to_numpy(dtype=float)

v = safe_numeric_series(analysis, "views")
l = safe_numeric_series(analysis, "likes")
d = safe_numeric_series(analysis, "dislikes")
c = safe_numeric_series(analysis, "comment_count")

# Safe ratios: where denominator is zero or invalid, output NaN.
like_denominator = l + d
like_ratio = np.divide(
    l, like_denominator,
    out=np.full_like(l, np.nan, dtype=float),
    where=np.isfinite(like_denominator) & (like_denominator > 0)
)
comment_rate = np.divide(
    c, v,
    out=np.full_like(c, np.nan, dtype=float),
    where=np.isfinite(v) & (v > 0)
)

def minmax_scale(array):
    finite = np.isfinite(array)
    result = np.full(array.shape, np.nan, dtype=float)
    if not finite.any():
        return result
    low = np.nanmin(array)
    high = np.nanmax(array)
    if high == low:
        result[finite] = 0.0
    else:
        result[finite] = (array[finite] - low) / (high - low)
    return result

views_norm = minmax_scale(v)
likes_norm = minmax_scale(l)
comments_norm = minmax_scale(c)
comment_rate_norm = minmax_scale(comment_rate)

# Missing component values contribute zero to the score, with missingness reported separately.
engagement_score = (
    0.35 * np.nan_to_num(views_norm, nan=0.0, posinf=0.0, neginf=0.0)
    + 0.25 * np.nan_to_num(likes_norm, nan=0.0, posinf=0.0, neginf=0.0)
    + 0.20 * np.nan_to_num(comments_norm, nan=0.0, posinf=0.0, neginf=0.0)
    + 0.10 * np.nan_to_num(like_ratio, nan=0.0, posinf=0.0, neginf=0.0)
    + 0.10 * np.nan_to_num(comment_rate_norm, nan=0.0, posinf=0.0, neginf=0.0)
)

# Log scaling: log1p is defined for zero and reduces the influence of very large values.
log_views = np.where(np.isfinite(v) & (v >= 0), np.log1p(v), np.nan)

analysis["like_ratio"] = like_ratio
analysis["comment_rate"] = comment_rate
analysis["views_norm"] = views_norm
analysis["likes_norm"] = likes_norm
analysis["comments_norm"] = comments_norm
analysis["comment_rate_norm"] = comment_rate_norm
analysis["engagement_score"] = engagement_score
analysis["log_views"] = log_views

# Boolean mask for the top 10% by engagement score.
top_10_cutoff = np.quantile(engagement_score, 0.90) if len(engagement_score) else np.nan
top_10_mask = engagement_score >= top_10_cutoff if len(engagement_score) else np.array([], dtype=bool)
analysis["top_10pct_engagement"] = top_10_mask

print("Like ratio sample:", like_ratio[:10])
print("Comment rate sample:", comment_rate[:10])
print("Engagement score sample:", engagement_score[:10])
print("90th-percentile engagement-score cutoff:", top_10_cutoff)
print("Rows flagged in top 10% mask:", int(top_10_mask.sum()))
print("Log-scaled view sample:", log_views[:10])

Like ratio sample: [0.92968076 0.94252874 0.92654028 0.89732143 0.9503866  0.98456075
 0.8216111  0.97410886 0.97298149 0.98387253]
Comment rate sample: [0.0056317  0.00739474 0.00408733 0.00834079 0.0010094  0.00227453
 0.00947871 0.00552818 0.00257922 0.00151743]
Engagement score sample: [0.1160525  0.12498149 0.11008285 0.12412998 0.0992218  0.1078879
 0.12108261 0.12108575 0.10848218 0.10515945]
90th-percentile engagement-score cutoff: 0.12298481450159109
Rows flagged in top 10% mask: 4600
Log-scaled view sample: [11.34267081 12.34772411 12.73332166 11.98851969 10.99275633 11.41871368
 11.70536236 13.09296134 12.66835592 12.66244314]


**Why log-scale views?** Video views are usually strongly right-skewed: a small number of videos can have far more views than most others. A raw-scale plot can compress the majority of records near zero; `log1p(views)` makes the distribution easier to inspect while still handling zero views.

## 9. Task 3 — Reshape, flatten and broadcast
We construct a daily trending-count matrix with **rows = days** and **columns = countries**. A country/day cell contains the number of distinct video records trending on that day. Then we use `reshape`, `flatten`, and broadcasting to compute each country's share of that day's total and each day's count relative to that country's average. The numerical operations use no explicit Python loops.

In [12]:
if "trending_date" not in analysis.columns or "country" not in analysis.columns:
    raise KeyError("Task 3 requires trending_date and country columns.")

daily_country_counts = (
    analysis.dropna(subset=["trending_date"])
    .groupby(["trending_date", "country"])
    .size()
    .unstack("country", fill_value=0)
    .sort_index()
)

daily_count_array = daily_country_counts.to_numpy(dtype=float)
print("Daily × country matrix shape:", daily_count_array.shape)

# Demonstrate reshape and flatten without changing the logical values.
reshaped_counts = daily_count_array.reshape(daily_count_array.shape[0], daily_count_array.shape[1])
flattened_counts = reshaped_counts.flatten()
print("Reshaped matrix shape:", reshaped_counts.shape)
print("Flattened array shape:", flattened_counts.shape)

# Broadcasting: each day's country count divided by that day's total.
daily_totals = daily_count_array.sum(axis=1, keepdims=True)
country_share_by_day = np.divide(
    daily_count_array, daily_totals,
    out=np.zeros_like(daily_count_array, dtype=float),
    where=daily_totals > 0
)

# Broadcasting: each day/country count relative to that country's average daily count.
country_daily_average = daily_count_array.mean(axis=0, keepdims=True)
relative_to_country_average = np.divide(
    daily_count_array, country_daily_average,
    out=np.full_like(daily_count_array, np.nan, dtype=float),
    where=country_daily_average > 0
)

country_share_df = pd.DataFrame(
    country_share_by_day, index=daily_country_counts.index, columns=daily_country_counts.columns
)
relative_average_df = pd.DataFrame(
    relative_to_country_average, index=daily_country_counts.index, columns=daily_country_counts.columns
)

print("Country shares by day (first 5 dates):")
display(country_share_df.head())
print("Daily count relative to each country's average (first 5 dates):")
display(relative_average_df.head())

Daily × country matrix shape: (80, 3)
Reshaped matrix shape: (80, 3)
Flattened array shape: (240,)
Country shares by day (first 5 dates):


country,GB,IN,US
trending_date,,,
2017-11-14,0.333333,0.333333,0.333333
2017-11-15,0.333333,0.333333,0.333333
2017-11-16,0.333333,0.333333,0.333333
2017-11-17,0.333333,0.333333,0.333333
2017-11-18,0.333333,0.333333,0.333333


Daily count relative to each country's average (first 5 dates):


country,GB,IN,US
trending_date,,,
2017-11-14,1.066667,1.066667,1.0
2017-11-15,1.066667,1.066667,1.0
2017-11-16,1.066667,1.066667,1.0
2017-11-17,1.066667,1.066667,1.0
2017-11-18,1.066667,1.066667,1.0


## 10. Task 4 — Inspect country DataFrames and combined data
We have already displayed `head()`, `info()`, `describe()` and `shape` for every source country and the combined raw data above. This compact summary makes row counts and shapes easy to compare.

In [13]:
country_shape_summary = pd.DataFrame([
    {"country": country, "rows": frame.shape[0], "columns": frame.shape[1]}
    for country, frame in country_dfs.items()
]).sort_values("country")
display(country_shape_summary)
print("Combined raw shape:", combined_raw.shape)
print("Combined cleaned shape:", clean.shape)

,country,rows,columns
0,GB,15075,17
1,IN,15075,17
2,US,16080,17


Combined raw shape: (46230, 17)
Combined cleaned shape: (46000, 17)


## 11. Task 5 — Filtering and sorting
These examples demonstrate boolean indexing with `&` and `|`, `loc`, `iloc`, and sorting. The displayed columns are restricted where useful for readability.

In [14]:
readable_cols = [
    c for c in [
        "country", "video_id", "title", "channel_title", "category_id",
        "views", "likes", "dislikes", "comment_count", "engagement_score", "trending_date"
    ] if c in analysis.columns
]

# (a) At least 1 million views AND more than 50,000 likes: boolean & and loc.
high_views_likes = analysis.loc[
    (analysis["views"] >= 1_000_000) & (analysis["likes"] > 50_000),
    readable_cols
].sort_values("views", ascending=False)
print("Records with 1M+ views AND >50K likes:", len(high_views_likes))
display(high_views_likes.head(50))

# (b) Top 20 records by engagement score.
top_20_engagement = analysis.sort_values("engagement_score", ascending=False).loc[:, readable_cols].head(20)
print("Top 20 records by engagement score:")
display(top_20_engagement)

# (c) Music OR Gaming category, with views above overall median.
# Category names are attached in Task 7; this section runs after the category merge.
print("Filtering for Music OR Gaming will be completed after category names are merged in Task 7.")

Records with 1M+ views AND >50K likes: 6718


,country,video_id,title,channel_title,category_id,views,likes,dislikes,comment_count,engagement_score,trending_date
10446,GB,RdqSDDng84n,Lena Ortiz - Monsoon (Live),FinalSpeech81,10,360760164.0,27537822.0,557330.0,598600.0,0.858557,2018-01-05
14874,GB,QYwpwP6y0Eb,The Hollow Pines - Phones (Official Video) WATCH,WildUnboxing39,10,343297002.0,9581910.0,258824.0,663209.0,0.695664,2018-01-27
10245,GB,RdqSDDng84n,Lena Ortiz - Monsoon (Live),FinalSpeech81,10,286714357.0,21885701.0,442939.0,475737.0,0.703838,2018-01-04
14673,GB,QYwpwP6y0Eb,The Hollow Pines - Phones (Official Video) WATCH,WildUnboxing39,10,282812154.0,7893691.0,213222.0,546360.0,0.591633,2018-01-26
8028,GB,WQEWsMmimzZ,Rook & Vale - Phones (Official Video) WATCH,PerfectTrailer65,10,255362869.0,11413793.0,618175.0,778390.0,0.658602,2017-12-24
10043,GB,RdqSDDng84n,Lena Ortiz - Monsoon (Live),FinalSpeech81,10,236275528.0,18035565.0,365017.0,392045.0,0.598447,2018-01-03
14472,GB,QYwpwP6y0Eb,The Hollow Pines - Phones (Official Video) WATCH,WildUnboxing39,10,230704137.0,6439282.0,173936.0,445693.0,0.502010,2018-01-25
9842,GB,RdqSDDng84n,Lena Ortiz - Monsoon (Live),FinalSpeech81,10,210747294.0,16086924.0,325579.0,349687.0,0.545106,2018-01-02
7827,GB,WQEWsMmimzZ,Rook & Vale - Phones (Official Video) WATCH,PerfectTrailer65,10,200444300.0,8959133.0,485230.0,610988.0,0.540024,2017-12-23
14272,GB,QYwpwP6y0Eb,The Hollow Pines - Phones (Official Video) WATCH,WildUnboxing39,10,181785943.0,5073905.0,137054.0,351189.0,0.417874,2018-01-24


Top 20 records by engagement score:


,country,video_id,title,channel_title,category_id,views,likes,dislikes,comment_count,engagement_score,trending_date
10446,GB,RdqSDDng84n,Lena Ortiz - Monsoon (Live),FinalSpeech81,10,360760164.0,27537822.0,557330.0,598600.0,0.858557,2018-01-05
10245,GB,RdqSDDng84n,Lena Ortiz - Monsoon (Live),FinalSpeech81,10,286714357.0,21885701.0,442939.0,475737.0,0.703838,2018-01-04
14874,GB,QYwpwP6y0Eb,The Hollow Pines - Phones (Official Video) WATCH,WildUnboxing39,10,343297002.0,9581910.0,258824.0,663209.0,0.695664,2018-01-27
8028,GB,WQEWsMmimzZ,Rook & Vale - Phones (Official Video) WATCH,PerfectTrailer65,10,255362869.0,11413793.0,618175.0,778390.0,0.658602,2017-12-24
10043,GB,RdqSDDng84n,Lena Ortiz - Monsoon (Live),FinalSpeech81,10,236275528.0,18035565.0,365017.0,392045.0,0.598447,2018-01-03
14673,GB,QYwpwP6y0Eb,The Hollow Pines - Phones (Official Video) WATCH,WildUnboxing39,10,282812154.0,7893691.0,213222.0,546360.0,0.591633,2018-01-26
9842,GB,RdqSDDng84n,Lena Ortiz - Monsoon (Live),FinalSpeech81,10,210747294.0,16086924.0,325579.0,349687.0,0.545106,2018-01-02
7827,GB,WQEWsMmimzZ,Rook & Vale - Phones (Official Video) WATCH,PerfectTrailer65,10,200444300.0,8959133.0,485230.0,610988.0,0.540024,2017-12-23
14472,GB,QYwpwP6y0Eb,The Hollow Pines - Phones (Official Video) WATCH,WildUnboxing39,10,230704137.0,6439282.0,173936.0,445693.0,0.502010,2018-01-25
9641,GB,RdqSDDng84n,Lena Ortiz - Monsoon (Live),FinalSpeech81,10,169181540.0,12914095.0,261365.0,280718.0,0.458255,2018-01-01


Filtering for Music OR Gaming will be completed after category names are merged in Task 7.


## 12. Task 6 — Missing values and duplicates (final report)
The missing-value table above reports the count and percentage for each affected column. The duplicate report states the key used and number removed. This final cell provides a concise audit summary suitable for adapting into the README.

In [15]:
print("MISSING VALUES")
display(missing_report if not missing_report.empty else pd.DataFrame({"message": ["No missing values found in combined raw data."]}))

print("\nDUPLICATES")
print(f"Duplicate key used: {duplicate_key if duplicate_key else 'identical full rows'}")
print(f"Duplicate rows removed: {duplicate_rows_removed:,}")
print(f"Rows remaining: {len(clean):,}")

MISSING VALUES


,missing_count,missing_percent
description,1624,3.51
comment_count,36,0.08
likes,12,0.03



DUPLICATES
Duplicate key used: ['video_id', 'trending_date', 'country']
Duplicate rows removed: 230
Rows remaining: 46,000


## 13. Task 7 — Merge category names
The category CSV can vary by dataset release: it may contain a JSON `category` column with a `title`, or it may already have category-name columns. The code below normalizes common formats and uses a **left merge** so every trending record is preserved. We compare row counts before and after the merge to demonstrate that no trending rows were lost.

In [16]:
category_path = DATA_DIR / "categories.csv"
if not category_path.exists():
    raise FileNotFoundError(f"categories.csv not found in {DATA_DIR}")

categories_raw = pd.read_csv(category_path, low_memory=False)
print("Categories file shape:", categories_raw.shape)
display(categories_raw.head())

cat = categories_raw.copy()
cat.columns = [str(c).strip() for c in cat.columns]

if "category_id" not in cat.columns:
    # Common alternate field name
    alternatives = [c for c in cat.columns if c.lower() in {"id", "categoryid", "category_id"}]
    if alternatives:
        cat = cat.rename(columns={alternatives[0]: "category_id"})

if "category_id" not in cat.columns:
    raise KeyError("categories.csv must include a category_id (or equivalent ID) column.")

if "category_title" not in cat.columns:
    title_candidates = [c for c in cat.columns if c.lower() in {"category_title", "title", "category_name", "name"}]
    if title_candidates:
        cat = cat.rename(columns={title_candidates[0]: "category_title"})

if "category_title" not in cat.columns and "category" in cat.columns:
    def extract_category_title(value):
        try:
            parsed = json.loads(value) if isinstance(value, str) else value
            if isinstance(parsed, dict):
                return parsed.get("title")
        except (json.JSONDecodeError, TypeError):
            return np.nan
        return np.nan
    cat["category_title"] = cat["category"].apply(extract_category_title)

if "category_title" not in cat.columns:
    raise KeyError("Could not find or extract category names from categories.csv.")

cat["category_id"] = pd.to_numeric(cat["category_id"], errors="coerce")
cat = cat[["category_id", "category_title"]].drop_duplicates(subset=["category_id"])

analysis_before_category_merge = len(analysis)
analysis = analysis.merge(cat, on="category_id", how="left", validate="many_to_one")
print("Merge type: left")
print("Rows before merge:", analysis_before_category_merge)
print("Rows after merge:", len(analysis))
print("Rows lost:", analysis_before_category_merge - len(analysis))
print("Rows with no matched category title:", int(analysis["category_title"].isna().sum()))
display(analysis[["category_id", "category_title"]].drop_duplicates().sort_values("category_id").head(30))

Categories file shape: (17, 2)


,category_id,category_name
0,1,Film & Animation
1,2,Autos & Vehicles
2,10,Music
3,15,Pets & Animals
4,17,Sports


Merge type: left
Rows before merge: 46000
Rows after merge: 46000
Rows lost: 0
Rows with no matched category title: 0


,category_id,category_title
23,1,Film & Animation
7,2,Autos & Vehicles
4,10,Music
84,15,Pets & Animals
16,17,Sports
97,19,Travel & Events
5,20,Gaming
6,22,People & Blogs
2,23,Comedy
3,24,Entertainment


### Finish Task 5(c): Music OR Gaming, above overall median views
This uses boolean `|` for the category choice and `&` for the view threshold.

In [17]:
overall_median_views = analysis["views"].median()
music_or_gaming = analysis.loc[
    analysis["category_title"].astype("string").str.casefold().isin(["music", "gaming"])
    & (analysis["views"] > overall_median_views),
    [c for c in ["country", "video_id", "title", "category_title", "views", "likes", "engagement_score"] if c in analysis.columns]
].sort_values("views", ascending=False)

print("Overall median views:", overall_median_views)
print("Music OR Gaming records above median views:", len(music_or_gaming))
display(music_or_gaming.head(100))

# Explicit iloc demonstration: first five rows and first few readable columns.
print("iloc example — first five rows and up to six columns:")
display(analysis.iloc[:5, :min(6, analysis.shape[1])])

Overall median views: 318389.0
Music OR Gaming records above median views: 9126


,country,video_id,title,category_title,views,likes,engagement_score
10400,GB,RdqSDDng84n,Lena Ortiz - Monsoon (Live),Music,360760164.0,27537822.0,0.858557
14800,GB,QYwpwP6y0Eb,The Hollow Pines - Phones (Official Video) WATCH,Music,343297002.0,9581910.0,0.695664
10200,GB,RdqSDDng84n,Lena Ortiz - Monsoon (Live),Music,286714357.0,21885701.0,0.703838
14601,GB,QYwpwP6y0Eb,The Hollow Pines - Phones (Official Video) WATCH,Music,282812154.0,7893691.0,0.591633
8000,GB,WQEWsMmimzZ,Rook & Vale - Phones (Official Video) WATCH,Music,255362869.0,11413793.0,0.658602
...,...,...,...,...,...,...,...
2809,GB,dzWtPHKYhEl,Rook & Vale - Guitar (Live),Music,41561719.0,1706646.0,0.173916
8208,GB,RdqSDDng84n,Lena Ortiz - Monsoon (Live),Music,41543928.0,3171163.0,0.191557
6207,GB,WQEWsMmimzZ,Rook & Vale - Phones (Official Video) WATCH,Music,41424361.0,1851518.0,0.196676
14404,GB,InFbaBprmWQ,Zed Rivera - Cars (Lyric Video),Music,41014722.0,1218740.0,0.166381


iloc example — first five rows and up to six columns:


,video_id,trending_date,title,channel_title,category_id,publish_time
0,u_nKn025TZS,2017-11-14,Wild Winter Prank,OfficialRecipe78,28,2017-10-18 07:42:02+00:00
1,FoQAEGxMMQD,2017-11-14,Wild Cricket Tutorial FINALE,FinalTrailer97,26,2017-11-14 12:56:18+00:00
2,wHqDBuStXq3,2017-11-14,Epic Space Challenge,UltimateReview54,23,2017-11-13 06:19:10+00:00
3,ZId0IGzSaaZ,2017-11-14,Quiet Paris Prank LIVE?!,FinalUpdate79,24,2017-11-14 11:37:33+00:00
4,1-xA4eRrYSH,2017-11-14,Nova Kane - Chess (Official Video)!,QuietInterview9,10,2017-11-14 03:13:26+00:00


## 14. Task 8 — Aggregation by category and country
We use `groupby()` with `agg()` for record count, average views, likes, and engagement score. `value_counts()` provides category counts and percentages. Publishing weekday/hour are derived from `publish_time`; the comparison uses average views.

In [18]:
category_agg = analysis.groupby("category_title", dropna=False).agg(
    record_count=("video_id", "size"),
    average_views=("views", "mean"),
    average_likes=("likes", "mean"),
    average_engagement_score=("engagement_score", "mean")
).sort_values("record_count", ascending=False)

country_agg = analysis.groupby("country", dropna=False).agg(
    record_count=("video_id", "size"),
    average_views=("views", "mean"),
    average_likes=("likes", "mean"),
    average_engagement_score=("engagement_score", "mean")
).sort_values("record_count", ascending=False)

print("By category:")
display(category_agg)
print("By country:")
display(country_agg)

category_counts = analysis["category_title"].value_counts(dropna=False)
category_distribution = pd.DataFrame({
    "count": category_counts,
    "percentage": (category_counts / category_counts.sum() * 100).round(2)
})
print("Category distribution (counts and percentages):")
display(category_distribution)

if "publish_time" in analysis.columns:
    analysis["publish_weekday"] = analysis["publish_time"].dt.day_name()
    analysis["publish_hour_utc"] = analysis["publish_time"].dt.hour
    weekday_views = analysis.groupby("publish_weekday")["views"].agg(["count", "mean", "median"]).sort_values("mean", ascending=False)
    hour_views = analysis.groupby("publish_hour_utc")["views"].agg(["count", "mean", "median"]).sort_values("mean", ascending=False)
    print("Publishing weekday by average views:")
    display(weekday_views)
    print("Publishing hour (UTC) by average views:")
    display(hour_views)
    print("Highest average-view weekday:", weekday_views["mean"].idxmax() if not weekday_views.empty else "N/A")
    print("Highest average-view hour (UTC):", hour_views["mean"].idxmax() if not hour_views.empty else "N/A")
else:
    print("publish_time is not available; publishing weekday/hour analysis skipped.")

By category:


,record_count,average_views,average_likes,average_engagement_score
category_title,,,,
Entertainment,12270,9.010256e+05,28939.305617,0.110809
Music,10310,3.190950e+06,134966.433838,0.108991
People & Blogs,3804,5.512341e+05,19835.408780,0.117689
Comedy,3258,6.034757e+05,28700.824071,0.112826
News & Politics,2422,2.837947e+05,3113.164463,0.104177
Film & Animation,2297,5.779740e+05,17191.443622,0.107386
Gaming,2057,5.096649e+05,20757.359922,0.116388
Sports,1921,3.749782e+05,10456.756250,0.106899
Howto & Style,1825,2.996848e+05,10222.452303,0.107771


By country:


,record_count,average_views,average_likes,average_engagement_score
country,,,,
US,16000,1.013904e+06,39560.292323,0.110439
GB,15000,1.584041e+06,65266.896439,0.110051
IN,15000,9.521008e+05,33144.305948,0.109745


Category distribution (counts and percentages):


...

Publishing weekday by average views:


,count,mean,median
publish_weekday,,,
Monday,6710,1.591747e+06,341062.0
Friday,6265,1.285801e+06,309931.0
Tuesday,7102,1.258620e+06,320612.5
Thursday,6600,1.110096e+06,310472.0
Wednesday,6547,1.101136e+06,322537.0
Saturday,6493,9.730517e+05,319033.0
Sunday,6283,9.129284e+05,305868.0


Publishing hour (UTC) by average views:


,count,mean,median
publish_hour_utc,,,
12,1859,3.244442e+06,338543.0
8,1853,2.080884e+06,324217.0
16,2046,1.521320e+06,335732.0
21,1847,1.431313e+06,336353.0
15,1975,1.350461e+06,296475.0
14,2032,1.309132e+06,297233.5
7,1967,1.240033e+06,351300.0
6,2028,1.227166e+06,337684.0
10,2109,1.193302e+06,332028.0


Highest average-view weekday: Monday
Highest average-view hour (UTC): 12


## 15. Task 9 — Time to trend and trending lifespan
`days_to_trend` is the difference in calendar days between publishing and trending dates. For lifespan, we count distinct trending dates per video and country, so a video trending in multiple countries has a separate country-level lifespan.

In [19]:
if all(c in analysis.columns for c in ["publish_time", "trending_date"]):
    analysis["publish_date"] = analysis["publish_time"].dt.tz_convert(None).dt.normalize()
    analysis["days_to_trend"] = (analysis["trending_date"] - analysis["publish_date"]).dt.days
    days_summary = analysis["days_to_trend"].describe(percentiles=[0.25, 0.5, 0.75, 0.9]).to_frame("days_to_trend")
    print("days_to_trend distribution (days):")
    display(days_summary)
    print("Negative days_to_trend rows (check date parsing/time zones):", int((analysis["days_to_trend"] < 0).sum()))
else:
    print("publish_time or trending_date is missing; cannot compute days_to_trend.")

lifespan_keys = [c for c in ["video_id", "country", "title", "channel_title", "category_title"] if c in analysis.columns]
if all(c in analysis.columns for c in ["video_id", "country", "trending_date"]):
    lifespan = (
        analysis.dropna(subset=["trending_date"])
        .groupby(["video_id", "country"], dropna=False)
        .agg(
            trending_lifespan_days=("trending_date", "nunique"),
            title=("title", "first") if "title" in analysis.columns else ("video_id", "first"),
            channel_title=("channel_title", "first") if "channel_title" in analysis.columns else ("video_id", "first"),
            category_title=("category_title", "first") if "category_title" in analysis.columns else ("video_id", "first"),
        )
        .reset_index()
        .sort_values(["trending_lifespan_days", "video_id"], ascending=[False, True])
    )
    print("10 longest-lasting video-country entries:")
    display(lifespan.head(10))
    print("Average lifespan per category:")
    lifespan_category = lifespan.groupby("category_title")["trending_lifespan_days"].mean().sort_values(ascending=False).to_frame("average_lifespan_days")
    display(lifespan_category)
else:
    print("video_id, country or trending_date missing; cannot compute trending lifespan.")

days_to_trend distribution (days):


,days_to_trend
count,46000.000000
mean,5.090457
std,6.243901
min,0.000000
25%,1.000000
50%,3.000000
75%,6.000000
90%,12.000000
max,47.000000


Negative days_to_trend rows (check date parsing/time zones): 0
10 longest-lasting video-country entries:


,video_id,country,trending_lifespan_days,title,channel_title,category_title
2989,BsTMdCTYs7P,US,25,Surprising Guitar Update,Brand NewUnboxing21,Entertainment
3379,DYLzrqG-SrK,GB,25,Official Football Update,InsaneReaction46,Entertainment
3627,EZaG9_2Yi1u,GB,25,KAIRO - FESTIVAL (LIVE),QuietChallenge75,Music
4381,Hh9b0BBsne0,GB,25,Kairo - Phones (Lyric Video)!,PerfectTrailer65,Music
5117,Kv1WBQp86cP,GB,25,Sunny Park - Cats (Live) MUST,FinalTrailer17,Music
5753,Nc1MUQvmyWO,US,25,Sunny Park - Cats (Official Video),ShockingInterview30,Music
5805,NqokCmZykzK,GB,25,Anya Lux - Paris (Lyric Video),QuietInterview9,Music
5869,O6zdCRwWdf1,GB,25,MIRA SOLIS - WINTER (LYRIC VIDEO)?,FinalTrailer17,Music
6657,RdqSDDng84n,GB,25,Lena Ortiz - Monsoon (Live),FinalSpeech81,Music
8411,Z1aFIGyR5zv,GB,25,Nova Kane - Robots (Official Video),QuietChallenge75,Music


Average lifespan per category:


,average_lifespan_days
category_title,
Music,5.333678
Entertainment,3.539083
Comedy,2.791774
People & Blogs,2.786813
Film & Animation,2.702353
Gaming,2.643959
Education,2.440653
Howto & Style,2.404480
Autos & Vehicles,2.369412


## 16. Task 10 — Per-video and per-channel summaries with `merge()`
We create a per-video summary, then a per-channel summary, and merge the summaries on channel. The requested output shows each channel's number of distinct trending videos and total views summed over the available trending records. Since the source data has one row per trending observation, “total views” here is the sum of views across observations, not a deduplicated lifetime view count.

In [20]:
required_channel_cols = {"channel_title", "video_id", "views"}
if required_channel_cols.issubset(analysis.columns):
    per_video_summary = analysis.groupby(
        ["country", "channel_title", "video_id"], dropna=False
    ).agg(
        video_max_views=("views", "max"),
        trending_observation_count=("video_id", "size"),
        average_engagement_score=("engagement_score", "mean")
    ).reset_index()

    per_channel_summary = analysis.groupby(
        ["country", "channel_title"], dropna=False
    ).agg(
        trending_video_count=("video_id", "nunique"),
        total_views_across_records=("views", "sum"),
        average_views_per_record=("views", "mean"),
        average_engagement_score=("engagement_score", "mean")
    ).reset_index()

    # Aggregate per-video summary to channel level, then merge with per-channel summary.
    per_video_channel = per_video_summary.groupby(
        ["country", "channel_title"], dropna=False
    ).agg(
        distinct_videos_from_video_summary=("video_id", "nunique"),
        summed_video_max_views=("video_max_views", "sum")
    ).reset_index()

    channel_comparison = per_channel_summary.merge(
        per_video_channel,
        on=["country", "channel_title"],
        how="left",
        validate="one_to_one"
    ).sort_values("total_views_across_records", ascending=False)

    print("Channel summary (top 20 by summed views across trending records):")
    display(channel_comparison.head(20))
    print("Rows in channel summary:", len(per_channel_summary))
    print("Rows after merge:", len(channel_comparison))
else:
    print("channel_title, video_id or views missing; channel summaries skipped.")

Channel summary (top 20 by summed views across trending records):


,country,channel_title,trending_video_count,total_views_across_records,average_views_per_record,average_engagement_score,distinct_videos_from_video_summary,summed_video_max_views
254,GB,QuietChallenge75,208,2.706290e+09,2.220090e+06,0.106220,208,601205990.0
381,GB,WildUnboxing39,23,2.447871e+09,1.261789e+07,0.126184,23,451867023.0
117,GB,FinalSpeech81,15,2.113776e+09,2.516400e+07,0.154996,15,381963541.0
245,GB,PerfectTrailer65,27,2.022323e+09,1.129789e+07,0.127735,27,390031997.0
1066,US,QuietChallenge75,126,1.544275e+09,2.412929e+06,0.106993,126,350280531.0
660,IN,QuietChallenge75,139,1.419200e+09,2.397297e+06,0.106464,139,388305518.0
1057,US,PerfectTutorial78,38,1.167257e+09,5.505929e+06,0.114723,38,237932199.0
258,GB,QuietInterview9,47,1.087882e+09,3.509296e+06,0.111574,47,216457109.0
240,GB,PerfectReaction36,25,1.046235e+09,6.082759e+06,0.113672,25,203916552.0
1131,US,SurprisingHighlights41,42,9.581560e+08,4.042852e+06,0.112779,42,195881648.0


Rows in channel summary: 1203
Rows after merge: 1203


## 17. Final checks and exportable summaries
This final section confirms the main acceptance criteria have corresponding outputs. It also saves cleaned data and summary tables locally as CSV files for optional inspection. These files are generated outputs; the committed source data remains unchanged.

In [21]:
print("FINAL ACCEPTANCE-CHECK SUMMARY")
print("- Country CSV files loaded:", len(country_dfs))
print("- Combined raw rows:", len(combined_raw))
print("- Combined raw columns:", combined_raw.shape[1])
print("- Duplicate rows removed:", duplicate_rows_removed)
print("- NumPy arrays reported:", ", ".join(arrays.keys()))
print("- Engagement-score weights: views 0.35, likes 0.25, comments 0.20, like ratio 0.10, comment rate 0.10")
print("- Category merge rows before/after:", analysis_before_category_merge, "→", len(analysis))
print("- Broadcasting matrix shape:", daily_count_array.shape)

# Optional local outputs for review; no external service is called.
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

analysis.to_csv(OUTPUT_DIR / "combined_cleaned_analysis.csv", index=False)
category_agg.to_csv(OUTPUT_DIR / "category_aggregation.csv")
country_agg.to_csv(OUTPUT_DIR / "country_aggregation.csv")
category_distribution.to_csv(OUTPUT_DIR / "category_distribution.csv")

if "lifespan" in globals():
    lifespan.to_csv(OUTPUT_DIR / "trending_lifespan.csv", index=False)
if "channel_comparison" in globals():
    channel_comparison.to_csv(OUTPUT_DIR / "channel_summary.csv", index=False)

print("Optional output CSVs saved in:", OUTPUT_DIR.resolve())

FINAL ACCEPTANCE-CHECK SUMMARY
- Country CSV files loaded: 3
- Combined raw rows: 46230
- Combined raw columns: 17
- Duplicate rows removed: 230
- NumPy arrays reported: views, likes, dislikes, comment_count
- Engagement-score weights: views 0.35, likes 0.25, comments 0.20, like ratio 0.10, comment rate 0.10
- Category merge rows before/after: 46000 → 46000
- Broadcasting matrix shape: (80, 3)
Optional output CSVs saved in: /content/outputs


## 18. README checklist
Before submitting, update `part2_numpy_pandas/README.md` with:
- Installation: `pip install numpy pandas`
- Run instructions and confirmation that all data is read from the local `data/` folder.
- Engagement-score formula and weight rationale.
- The actual missing-value counts and percentages printed by the notebook.
- The duplicate definition and exact number removed.
- Other observed quality issues (invalid dates, removed/private videos, negative counts, unmatched category IDs) and how they were handled.
- A short business interpretation of category/country results, publishing patterns, days to trend and trending lifespan.

Do not claim that no records were lost in the category merge unless the printed before/after row counts confirm it.